# Run the Credit Score tutorial

We'll use the following tutorial as a demonstration.

https://github.com/feast-dev/feast-credit-score-local-tutorial

## Upload the tutorial source code to the running Feast pod.

Verify the client `feature_store.yaml`.

In [1]:
![ -f f43b44b.tar.gz ] || wget https://github.com/feast-dev/feast-credit-score-local-tutorial/archive/f43b44b.tar.gz
!kubectl cp f43b44b.tar.gz $(kubectl get pods -l 'feast.dev/name=example' -ojsonpath="{.items[*].metadata.name}"):/feast-data -c registry
!kubectl exec deploy/feast-example -c registry -- rm -rf /feast-data/feast-credit-score-local-tutorial
!kubectl exec deploy/feast-example -c registry -- mkdir /feast-data/feast-credit-score-local-tutorial
!kubectl exec deploy/feast-example -c registry -- tar vfx /feast-data/f43b44b.tar.gz -C /feast-data/feast-credit-score-local-tutorial --strip-components 1

--2025-01-14 08:29:52--  https://github.com/feast-dev/feast-credit-score-local-tutorial/archive/f43b44b.tar.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/feast-dev/feast-credit-score-local-tutorial/tar.gz/f43b44b245ae2632b582f14176392cfe31f98da9 [following]
--2025-01-14 08:29:52--  https://codeload.github.com/feast-dev/feast-credit-score-local-tutorial/tar.gz/f43b44b245ae2632b582f14176392cfe31f98da9
Resolving codeload.github.com (codeload.github.com)... 140.82.114.10
Connecting to codeload.github.com (codeload.github.com)|140.82.114.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘f43b44b.tar.gz’

f43b44b.tar.gz          [    <=>             ]  45.52M   189KB/s    in 2m 14s  

2025-01-14 08:32:06 (348 KB/s) - ‘f43b44b.tar.gz’ saved [47734189]

feast-cr

## Verify the client `feature_store.yaml`.

Move the `feature_store.yaml` config to demo directory and verify its contents.

In [2]:
!kubectl exec deploy/feast-example -c registry -- [ -f /feast-data/feature_store.yaml ] || kubectl exec deploy/feast-example -c registry -- mv /feast-data/credit_scoring_local/feature_repo/feature_store.yaml /feast-data/feature_store.yaml
!kubectl exec deploy/feast-example -c registry -- cp -f /feast-data/feature_store.yaml /feast-data/feast-credit-score-local-tutorial/feature_repo/feature_store.yaml
!kubectl exec deploy/feast-example -c registry -- cat /feast-data/feast-credit-score-local-tutorial/feature_repo/feature_store.yaml

command terminated with exit code 1
project: credit_scoring_local
provider: local
offline_store:
    type: duckdb
online_store:
    type: redis
    connection_string: redis.feast.svc.cluster.local:6379
registry:
    path: postgresql+psycopg://postgres@postgres.feast.svc.cluster.local:5432/feast
    registry_type: sql
    cache_ttl_seconds: 60
    sqlalchemy_config_kwargs:
        echo: false
        pool_pre_ping: true
auth:
    type: no_auth
entity_key_serialization_version: 3


## Apply the feature store definitions

In [3]:
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo apply

/usr/local/lib/python3.11/site-packages/feast/feature_store.py:575: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
No project found in the repository. Using project name credit_scoring_local defined in feature_store.yaml
Applying changes for project credit_scoring_local
Deploying infrastructure for credit_history
Deploying infrastructure for zipcode_features


In [4]:
!kubectl exec deploy/feast-example -c registry -- bash -c 'cd /feast-data/feast-credit-score-local-tutorial/feature_repo && feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")'

Materializing 2 feature views to 2025-01-09 19:06:55+00:00 into the redis online store.

credit_history from 2024-10-11 19:07:29+00:00 to 2025-01-09 19:06:55+00:00:
0it [00:00, ?it/s]
100%|███████████████████████████████████████████████████████| 28844/28844 [00:22<00:00, 1292.46it/s]


## Execute feast commands inside the client Pod

Finally, we run the full test suite from the client folder.

In [5]:
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo feature-views list

NAME              ENTITIES     TYPE
credit_history    {'dob_ssn'}  FeatureView
zipcode_features  {'zipcode'}  FeatureView
total_debt_calc   {'dob_ssn'}  OnDemandFeatureView
spec:
  name: total_debt_calc
  features:
  - name: total_debt_due
    valueType: DOUBLE
  sources:
    application_data:
      requestDataSource:
        type: REQUEST_SOURCE
        requestDataOptions:
          schema:
          - name: loan_amnt
            valueType: INT64
        name: application_data
    credit_history:
      featureViewProjection:
        featureViewName: credit_history
        featureColumns:
        - name: credit_card_due
          valueType: INT64
        - name: mortgage_due
          valueType: INT64
        - name: student_loan_due
          valueType: INT64
        - name: vehicle_loan_due
          valueType: INT64
        - name: hard_pulls
          valueType: INT64
        - name: missed_payments_2y
          valueType: INT64
        - name: missed_payments_1y
          valueTyp

## Run test code inside the client Pod

Finally, we run the full test suite from the client folder.

In [6]:
!kubectl exec deploy/feast-example -c registry -- python -m venv --system-site-packages /feast-data/venv
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && pip install -r /feast-data/feast-credit-score-local-tutorial/requirements.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.2/540.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.2/326.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 M

In [7]:
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && cd /feast-data/feast-credit-score-local-tutorial && python run.py'

Loan rejected!


In [ ]:
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && cd /feast-data/feast-credit-score-local-tutorial && streamlit run --server.port 8501 streamlit_app.py'




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.42.0.8:8501
  External URL: http://23.112.66.217:8501

2025-01-09 19:23:08.950 
Calling `st.pyplot()` without providing a figure argument has been deprecated
and will be removed in a later version as it requires the use of Matplotlib's
global figure object, which is not thread-safe.

To future-proof this code, you should pass in a figure as shown below:

```python
fig, ax = plt.subplots()
ax.scatter([1, 2, 3], [1, 2, 3])
# other plotting actions...
st.pyplot(fig)
```

If you have a specific use case that requires this functionality, please let us
know via [issue on Github](https://github.com/streamlit/streamlit/issues).

2025-01-09 19:23:16.307 
Calling `st.pyplot()` without providing a figure argument has been deprecated
and will be removed in a later version as it requires the use of Matplotlib's
global figure object, which is not thread-safe.

To future-proof this c

```bash
$ kubectl port-forward deploy/feast-example 8501:8501
```
Then navigate to the local URL on which Streamlit is being served.

http://localhost:8501